# 1. Train the grain-type classifier

Trains the 1D CNN on the labelled SMP profiles and evaluates it on a held-out
test split.

In [ ]:
import sys
sys.path.append("../src")

import pickle
import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader

import config
import data
import evaluation
import inference
import model
import plotting
import training

## Load the labelled profiles

The stored labels use five classes; `merge_to_four_classes` combines
*Fragmented & Rounded* with *Faceted*, which the force signal alone cannot
reliably separate.

In [ ]:
profiles = data.merge_to_four_classes(xr.open_dataset(config.LABELLED_DATA))

print(f"profiles:   {profiles.sizes['profile']}")
print(f"depth bins: {profiles.sizes['depth_bins']}")

labels = profiles.dominant_label.values.flatten()
labels = labels[~np.isnan(labels)]
for i, name in enumerate(config.CLASS_NAMES):
    count = int((labels == i).sum())
    print(f"{i}  {name:<20} {count:>7,} bins  ({count / len(labels):5.1%})")

## Split, normalize, weight

Whole profiles are split, never individual depth bins. Normalization constants
and class weights come from the training profiles only.

In [ ]:
train_ids, val_ids, test_ids = data.split_profiles(
    profiles, val_fraction=0.2, test_fraction=0.1)
print(f"train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)} profiles")

norm = data.compute_normalization_constants(profiles, train_ids)
class_weights = data.compute_class_weights(profiles, train_ids)

for i, name in enumerate(config.CLASS_NAMES):
    print(f"{name:<20} weight {class_weights[i]:.4f}")

In [ ]:
train_set = data.SMPProfileDataset(profiles, train_ids, norm)
val_set = data.SMPProfileDataset(profiles, val_ids, norm)
test_set = data.SMPProfileDataset(profiles, test_ids, norm)

train_loader = DataLoader(train_set, batch_size=config.BATCH_SIZE, shuffle=True,
                          collate_fn=data.pad_and_collate)
val_loader = DataLoader(val_set, batch_size=config.BATCH_SIZE,
                        collate_fn=data.pad_and_collate)
test_loader = DataLoader(test_set, batch_size=config.BATCH_SIZE,
                         collate_fn=data.pad_and_collate)

features, labels = train_set[0]
print(f"one profile -> features {tuple(features.shape)}, labels {tuple(labels.shape)}")

## Train

The best-scoring epoch is checkpointed, not the last one.

To log to Weights & Biases, create a run and pass it as `run=`; without it wandb
is never imported.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

net = model.CNN1D()
history = training.train_model(net, train_loader, val_loader,
                               class_weights=class_weights,
                               epochs=config.NUM_EPOCHS)

print(f"\nbest validation accuracy {history['best_val_accuracy']:.4f}")
print(f"checkpoint {history['checkpoint_path']}")

In [ ]:
plotting.plot_training_history(history)
plt.show()

## Evaluate on the held-out test profiles

These were never seen during training or model selection.

In [ ]:
net.load_state_dict(torch.load(history["checkpoint_path"]))
metrics = evaluation.evaluate_model(
    net, test_loader, nn.CrossEntropyLoss(ignore_index=config.PADDING_LABEL))

for name in ["loss", "accuracy", "precision", "recall", "f1"]:
    print(f"{name:<10} {metrics[name]:.4f}")

In [ ]:
plotting.plot_confusion_matrix(metrics["confusion_matrix"])
plt.show()

## Save the constants alongside the weights

The weights alone are not enough to reproduce a prediction: inference has to
reuse these exact normalization constants.

In [ ]:
with open(history["checkpoint_path"].parent / "normalization_constants.pkl", "wb") as f:
    pickle.dump(norm, f)

print("weights  ", history["checkpoint_path"])
print("constants", history["checkpoint_path"].parent / "normalization_constants.pkl")